In [1]:
import numpy as np
import pandas as pd

In [2]:
%load_ext sql


In [3]:
%sql sqlite:///data/basic_examples.db --alias sqlite

Connecting to 'sqlite'

In [4]:
%sql duckdb:///data/example_duck.db --alias duckdb
%sql set python_scan_all_frames = True;

Connecting and switching to connection 'duckdb'

Running query in 'duckdb'

Success


In [5]:
%%sql sqlite
SELECT * FROM sqlite_master WHERE type='table'

Switching to connection 'sqlite'

type,name,tbl_name,rootpage,sql
table,sqlite_sequence,sqlite_sequence,7,"CREATE TABLE sqlite_sequence(name,seq)"
table,Dragon,Dragon,2,"CREATE TABLE Dragon ( name TEXT PRIMARY KEY, year INTEGER CHECK (year >= 2000), cute INTEGER)"
table,Dish,Dish,4,"CREATE TABLE Dish ( name TEXT PRIMARY KEY, type TEXT, cost INTEGER CHECK (cost >= 0))"
table,Scene,Scene,6,"CREATE TABLE Scene ( id INTEGER PRIMARY KEY AUTOINCREMENT, biome TEXT NOT NULL, city TEXT NOT NULL, visitors INTEGER CHECK (visitors >= 0), created_at DATETIME DEFAULT (DATETIME('now')))"


In [6]:
%%sql duckdb
DROP TABLE IF EXISTS dragon;
DROP TABLE IF EXISTS dish;
DROP TABLE IF EXISTS setting;
DROP TABLE IF EXISTS scene;

CREATE TABLE setting (
    id INTEGER PRIMARY KEY,
    media VARCHAR NOT NULL,
    place VARCHAR,
    type VARCHAR,
    start_year INT CHECK (start_year >= 1900)
);

CREATE TABLE dragon (
    name VARCHAR PRIMARY KEY,
    yr INTEGER CHECK (yr >= 1900),
    cute INTEGER,
    setting_id INTEGER,
    FOREIGN KEY (setting_id) REFERENCES setting(id)
);

CREATE TABLE dish (
    name VARCHAR PRIMARY KEY,
    type VARCHAR,
    cost INTEGER CHECK (cost >= 0)
);

CREATE TABLE scene (
    id INTEGER PRIMARY KEY,
    biome VARCHAR NOT NULL,
    city VARCHAR NOT NULL,
    visitors INTEGER CHECK (visitors >= 0),
    created_at DATETIME DEFAULT current_date()
);

Switching to connection 'duckdb'

Count


In [7]:
import pandas as pd
pydragon = pd.DataFrame({
    'name': ['toothless', 'drogon', 'dragon 2', 'puff', 'smaug', 'rhaegal'],
    'yr':   [2010,        2011,     2019,       1963,   1937,    2011     ],
    'cute': [10.0,        -100.0,   0.0,        100.0,  pd.NA,   -80.0   ],
    'setting_id': [0,     1,        2,          3,      4,       1        ],  
})
pysetting = pd.DataFrame({
    'id': [0, 1, 2, 3, 4],
    'media':      ['How to Train Your Dragon', 'Game of Thrones', 'IRL', 'Puff the Magic Dragon', 'The Hobbit'],
    'place':      ['Berk',                     'Essos',           pd.NA, 'Honalee',               'Lonely Mountain'], 
    'type':       ['movie',                    'tv show',         'irl', 'song',                  'book'],
    'start_year': [2010,                        2011,             pd.NA, 1963,                    1937],
})
pydish = pd.DataFrame({
    'name': ['ravioli', 'ramen', 'taco', 'edamame', 'fries', 'potsticker', 'ice cream'],
    'type': ['entree', 'entree', 'entree', 'appetizer', 'appetizer', 'appetizer', 'dessert'],
    'cost': [10, 13, 7, 4, 4, 4, 5],
})
pyscene = pd.read_csv('data/scene.csv')


In [8]:
%%sql duckdb
INSERT INTO setting (SELECT * FROM pysetting);
INSERT INTO dragon (SELECT * FROM pydragon);
INSERT INTO dish (SELECT * FROM pydish);
INSERT INTO scene (SELECT * FROM pyscene);

COMMIT;
CHECKPOINT;

Success


In [9]:
%%sql duckdb

COMMIT;
CHECKPOINT;

Success


Closing sqlite

Closing duckdb